# 06. Final Data Comparison & Integration
## Compare raw consolidated vs processed dataset

**Objective**:
- Compare original consolidated dataset with processed dataset
- Document all transformations applied
- Create final integrated dataset ready for analysis

**Output**:
- Side-by-side comparison report
- Final integrated dataset with all transformations documented

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Load all versions
data_dir = Path('./outputs')
consolidated_raw = pd.read_csv(data_dir / 'consolidated_raw_data.csv', index_col=0)
consolidated_scaled = pd.read_csv(data_dir / 'consolidated_scaled.csv', index_col=0)

print(f"✓ Loaded datasets:")
print(f"  Raw: {consolidated_raw.shape}")
print(f"  Scaled: {consolidated_scaled.shape}")

## 1. Data Comparison Report

In [ ]:
# Comparison report
print("\n" + "="*80)
print("DATA TRANSFORMATION SUMMARY")
print("="*80 + "\n")

comparison = {
    'Stage': ['Raw Consolidated', 'Outlier Detected', 'Log Transformed', 'KNN Imputed', 'StandardScaled'],
    'Missing_Values': [
        consolidated_raw.isna().sum().sum(),
        consolidated_raw.isna().sum().sum(),  # No change in outlier detection
        consolidated_raw.isna().sum().sum(),  # No change in log transform
        0,  # After KNN
        0   # After scaling
    ],
    'Data_Quality': [
        'Raw data from World Bank',
        'Outliers flagged (not removed)',
        'Log transform for skewed distributions',
        'KNN imputation (k=5)',
        'StandardScaler normalization'
    ]
}

comparison_df = pd.DataFrame(comparison)
print(comparison_df.to_string(index=False))

# Statistics comparison
print("\n" + "-"*80)
print("STATISTICS COMPARISON - Vietnam (VNM) Sample:")
print("-"*80 + "\n")

if 'VNM' in consolidated_raw.index:
    print("Raw - First 5 columns:")
    print(consolidated_raw.loc['VNM'].head(5).to_string())
    
    print("\n" + "-"*40)    
    print("\nScaled - First 5 columns (only data, exclude cluster):")
    print(consolidated_scaled.loc['VNM', :-1].head(5).to_string())

## 2. Create Final Integrated Dataset

In [ ]:
# Create final integrated dataset with metadata
print("\n" + "="*80)
print("CREATING FINAL INTEGRATED DATASET")
print("="*80 + "\n")

# Load all components
outlier_flags = pd.read_csv(data_dir / 'outlier_flags.csv', index_col=0)
cluster_assignments = pd.read_csv(data_dir / 'cluster_assignments.csv', index_col=0)

# Combine: Raw + Scaled + Flags + Clusters (with suffix)
final_dataset = pd.DataFrame(index=consolidated_raw.index)

# Add raw data
for col in consolidated_raw.columns:
    final_dataset[f'{col}_raw'] = consolidated_raw[col]

# Add scaled data (exclude cluster column from scaled)
scaled_cols = [c for c in consolidated_scaled.columns if c != 'cluster']
for col in scaled_cols:
    final_dataset[f'{col}_scaled'] = consolidated_scaled[col]

# Add outlier flags
for col in outlier_flags.columns:
    final_dataset[f'{col}_outlier_flag'] = outlier_flags[col]

# Add cluster assignment
final_dataset['cluster'] = cluster_assignments['cluster']

print(f"✓ Final integrated dataset created: {final_dataset.shape}")
print(f"\n Columns:")
print(f"  Raw data (_raw): {len([c for c in final_dataset.columns if '_raw' in c])}")
print(f"  Scaled data (_scaled): {len([c for c in final_dataset.columns if '_scaled' in c])}")
print(f"  Outlier flags (_outlier_flag): {len([c for c in final_dataset.columns if '_outlier' in c])}")
print(f"  Cluster assignment: 1")
print(f"\n Total columns: {final_dataset.shape[1]}")

## 3. Save Final Integrated Dataset

In [ ]:
# Save final dataset
final_dataset.to_csv(data_dir / 'final_integrated_dataset.csv')

# Create summary metadata
metadata = {
    'dataset_name': 'World Bank Data - Integrated & Processed',
    'total_rows': len(final_dataset),
    'total_columns': final_dataset.shape[1],
    'indicators': 8,
    'time_range': '2004-2024',
    'data_transformations': [
        'Outlier detection (IQR method, flagged)',
        'Log transformation (handle negative/zero)',
        'KNN imputation (k=5)',
        'StandardScaler normalization',
        'K-Means clustering (k=6)'
    ],
    'column_categories': {
        'raw': 'Original World Bank values (_raw suffix)',
        'scaled': 'Normalized scaled values (_scaled suffix)',
        'outlier_flag': 'Binary outlier indicators (_outlier_flag suffix)',
        'cluster': 'K-Means cluster assignment'
    },
    'missing_handling': 'KNN imputation (k=5) + forward/backward fill fallback',
    'vietnam_status': 'VNM included with peer group analysis'
}

with open(data_dir / 'final_dataset_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ Saved:")
print(f"  final_integrated_dataset.csv ({final_dataset.shape[0]} × {final_dataset.shape[1]})")
print(f"  final_dataset_metadata.json")
print(f"\n✓ DATA INTEGRATION COMPLETE")
print(f"\nAll processed data files ready in: {data_dir}")